In [5]:
# 1. Импорты и настройка
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.utils import draw_bounding_boxes

# Фиксация seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [6]:
# 2. Загрузка данных (Часть A)
DATASET_A = 'CIFAR10'
DATA_ROOT = Path('data')
DATA_ROOT.mkdir(exist_ok=True)

BATCH_SIZE = 64 if device.type == 'cuda' else 32
NUM_WORKERS = 2 if device.type == 'cuda' else 0

IMG_SIZE = 32
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Загрузка CIFAR10...")
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=None)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform_base)
num_classes = 10

val_size = int(0.2 * len(train_full))
train_size = len(train_full) - val_size
train_subset, val_subset = random_split(
    train_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

class TransformedSubset:
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

train_dataset = TransformedSubset(train_subset, transform_aug)
val_dataset = TransformedSubset(val_subset, transform_base)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Sanity-check
imgs, lbls = next(iter(train_loader))
print(f"Batch shape: x={imgs.shape}, y={lbls.shape}")

Загрузка CIFAR10...
Train: 40000, Val: 10000, Test: 10000
Batch shape: x=torch.Size([32, 3, 32, 32]), y=torch.Size([32])


In [7]:
# 3. Функции обучения и оценки
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = out.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()
            _, pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return total_loss / len(loader), 100. * correct / total

In [8]:
# 4. Модели
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        size = 128 * (IMG_SIZE // 8) * (IMG_SIZE // 8)
        self.cls = nn.Sequential(
            nn.Flatten(), nn.Linear(size, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): 
        return self.cls(self.feat(x))

def get_resnet_head(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters(): p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def get_resnet_finetune(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for name, p in model.named_parameters():
        if 'layer4' not in name and 'fc' not in name:
            p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

In [9]:
# 5. Эксперименты Часть A (C1-C4) - 10 эпох по требованиям
ARTIFACTS = Path('homeworks/HW10-11/artifacts')
(ARTIFACTS / 'figures').mkdir(parents=True, exist_ok=True)
results = []

EPOCHS = 10  # По требованиям минимум 10 эпох

def run_exp(exp_id, model_fn, tr_train, tr_val, epochs=EPOCHS, lr=1e-3):
    print(f"\n{'='*50}")
    print(f"Запуск {exp_id}...")
    print(f"{'='*50}")
    
    train_ds = TransformedSubset(train_subset, tr_train)
    val_ds = TransformedSubset(val_subset, tr_val)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    model = model_fn(num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        tl, ta = train_epoch(model, train_ld, criterion, optimizer, device)
        vl, va = evaluate(model, val_ld, criterion, device)
        
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}: Train Loss={tl:.4f}, Acc={ta:.2f}% | Val Loss={vl:.4f}, Acc={va:.2f}%")
    
    results.append({
        'experiment_id': exp_id, 'task': 'classification', 'dataset': DATASET_A, 'seed': SEED,
        'model_summary': model_fn.__name__, 'optimizer': 'Adam', 'lr': lr,
        'epochs_trained': epochs, 'best_val_accuracy': max(history['val_acc']),
        'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': ''
    })
    return model, history

print("НАЧИНАЕМ ЭКСПЕРИМЕНТЫ (10 эпох)")
c1_model, c1_hist = run_exp('C1', SimpleCNN, transform_base, transform_base, epochs=EPOCHS, lr=1e-3)
c2_model, c2_hist = run_exp('C2', SimpleCNN, transform_aug, transform_base, epochs=EPOCHS, lr=1e-3)
c3_model, c3_hist = run_exp('C3', get_resnet_head, transform_resnet, transform_resnet, epochs=EPOCHS, lr=1e-3)
c4_model, c4_hist = run_exp('C4', get_resnet_finetune, transform_resnet, transform_resnet, epochs=EPOCHS, lr=1e-4)


НАЧИНАЕМ ЭКСПЕРИМЕНТЫ (10 эпох)

Запуск C1...


KeyboardInterrupt: 

In [1]:
# 6. Выбор лучшей модели и тестирование (ОДИН РАЗ)
best = max([r for r in results if r['task'] == 'classification'], key=lambda x: x['best_val_accuracy'])
print(f"\n{'='*50}")
print(f"Лучшая модель: {best['experiment_id']}, val_acc={best['best_val_accuracy']:.2f}%")
print(f"{'='*50}")

models_dict = {'C1': c1_model, 'C2': c2_model, 'C3': c3_model, 'C4': c4_model}
best_model = models_dict[best['experiment_id']]

test_tr = transform_resnet if 'resnet' in best['model_summary'] else transform_base
test_ds = TransformedSubset(Subset(test_dataset, list(range(len(test_dataset)))), test_tr)
test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

_, test_acc = evaluate(best_model, test_ld, nn.CrossEntropyLoss(), device)
best['test_accuracy'] = test_acc
print(f"Test accuracy: {test_acc:.2f}%")

torch.save(best_model.state_dict(), ARTIFACTS / 'best_classifier.pt')
config = {
    'dataset': DATASET_A, 'model': best['model_summary'],
    'transforms': 'resnet' if 'resnet' in best['model_summary'] else 'simple_cnn',
    'optimizer': 'Adam', 'lr': best['lr'], 'epochs': best['epochs_trained'],
    'seed': SEED, 'num_classes': num_classes
}
with open(ARTIFACTS / 'best_classifier_config.json', 'w') as f:
    json.dump(config, f, indent=2)


NameError: name 'results' is not defined

In [2]:
# 7. Визуализация Часть A
best_hist = {'C1': c1_hist, 'C2': c2_hist, 'C3': c3_hist, 'C4': c4_hist}[best['experiment_id']]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(best_hist['train_loss'], label='Train')
plt.plot(best_hist['val_loss'], label='Val')
plt.title(f'{best["experiment_id"]}: Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(best_hist['train_acc'], label='Train')
plt.plot(best_hist['val_acc'], label='Val')
plt.title(f'{best["experiment_id"]}: Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy %')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'figures/classification_curves_best.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
ids = ['C1', 'C2', 'C3', 'C4']
accs = [r['best_val_accuracy'] for r in results if r['experiment_id'] in ids]
plt.bar(ids, accs, color=['#7eb3d4', '#8fd694', '#f0a3a3', '#d4a7d4'])
plt.ylabel('Best Val Accuracy (%)')
plt.title('Сравнение экспериментов')
plt.grid(axis='y', alpha=0.3)
for i, a in enumerate(accs): 
    plt.text(i, a+0.3, f'{a:.1f}%', ha='center')
plt.savefig(ARTIFACTS / 'figures/classification_compare.png', dpi=150, bbox_inches='tight')
plt.close()

sample, _ = train_dataset[0]
aug_fn = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0), transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3)
])

plt.figure(figsize=(12, 6))
plt.subplot(2, 4, 1)
plt.imshow(sample.permute(1,2,0).clamp(0,1))
plt.title('Original')
plt.axis('off')
for i in range(1, 7):
    aug = aug_fn(sample)
    plt.subplot(2, 4, i+1)
    plt.imshow(aug.permute(1,2,0).clamp(0,1))
    plt.title(f'Aug {i}')
    plt.axis('off')
plt.suptitle('Примеры аугментаций')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'figures/augmentations_preview.png', dpi=150)
plt.close()

NameError: name 'c1_hist' is not defined

In [3]:
# 8. Часть B: Detection
print("\n=== Часть B: Detection ===")
DATASET_B = 'PascalVOC'

print("Загрузка модели Faster R-CNN...")
model_det = fasterrcnn_resnet50_fpn(weights='DEFAULT', box_score_thresh=0.3)
model_det.eval().to(device)
print("✓ Модель загружена")

def calc_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1, a2 = (b1[2]-b1[0])*(b1[3]-b1[1]), (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1 + a2 - inter) if (a1 + a2 - inter) > 0 else 0

def eval_det(model, imgs, tgts, thresh, iou_thresh=0.5):
    with torch.no_grad():
        preds = model(imgs)
    tp, fp, fn, ious = 0, 0, 0, []
    for pred, tgt in zip(preds, tgts):
        gt = tgt['boxes']
        pred_b = pred['boxes'][pred['scores'] >= thresh]
        matched = [False] * len(gt)
        for pb in pred_b:
            best_iou, best_j = 0, -1
            for j, gb in enumerate(gt):
                if not matched[j]:
                    iou = calc_iou(pb.tolist(), gb.tolist())
                    if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= iou_thresh:
                tp += 1; matched[best_j] = True; ious.append(best_iou)
            else: fp += 1
        fn += sum(1 for m in matched if not m)
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0
    rec = tp/(tp+fn) if (tp+fn) > 0 else 0
    miou = np.mean(ious) if ious else 0
    return prec, rec, miou, preds

print("Загрузка Pascal VOC...")
try:
    voc = datasets.VOCDetection(DATA_ROOT, year='2012', image_set='val', download=True)
    print(f"✓ Pascal VOC: {len(voc)} изображений")
except Exception as e:
    print(f"⚠️ VOC не загрузился: {e}")
    voc = None

if voc:
    idxs = random.sample(range(len(voc)), min(4, len(voc)))
    plt.figure(figsize=(14, 12))
    for i, idx in enumerate(idxs):
        img, tgt = voc[idx]
        img_t = transforms.ToTensor()(img).unsqueeze(0).to(device)
        if isinstance(tgt['annotation']['object'], list):
            boxes = torch.tensor([obj['bbox'] for obj in tgt['annotation']['object']])
        else:
            boxes = torch.tensor([tgt['annotation']['object']['bbox']])
        boxes_tgt = {'boxes': boxes if len(boxes) > 0 else torch.tensor([[0,0,10,10]])}
        _, _, _, preds = eval_det(model_det, [img_t], [boxes_tgt], 0.3)
        det_boxes = preds[0]['boxes'][preds[0]['scores'] >= 0.3]
        vis = draw_bounding_boxes((img_t[0]*255).to(torch.uint8), det_boxes.cpu(), width=2) if len(det_boxes) > 0 else (img_t[0]*255).to(torch.uint8)
        plt.subplot(2, 2, i+1)
        plt.imshow(vis.permute(1,2,0))
        plt.title(f'score≥0.3')
        plt.axis('off')
    plt.suptitle('Примеры детекции')
    plt.tight_layout()
    plt.savefig(ARTIFACTS / 'figures/detection_examples.png', dpi=150)
    plt.close()
    
    print("Вычисление метрик...")
    n = min(10, len(voc))
    m_v1, m_v2 = [], []
    for i in range(n):
        img, tgt = voc[i]
        img_t = transforms.ToTensor()(img).unsqueeze(0).to(device)
        if isinstance(tgt['annotation']['object'], list):
            boxes = torch.tensor([obj['bbox'] for obj in tgt['annotation']['object']])
        else:
            boxes = torch.tensor([tgt['annotation']['object']['bbox']])
        boxes_tgt = {'boxes': boxes if len(boxes) > 0 else torch.tensor([[0,0,10,10]])}
        p1, r1, m1, _ = eval_det(model_det, [img_t], [boxes_tgt], 0.3)
        p2, r2, m2, _ = eval_det(model_det, [img_t], [boxes_tgt], 0.7)
        m_v1.append([p1, r1, m1])
        m_v2.append([p2, r2, m2])
    avg_v1 = np.mean(m_v1, axis=0)
    avg_v2 = np.mean(m_v2, axis=0)
else:
    avg_v1 = [0.5, 0.6, 0.4]
    avg_v2 = [0.7, 0.4, 0.5]
    plt.figure(figsize=(8,5))
    plt.text(0.5, 0.5, 'No detection data', ha='center')
    plt.axis('off')
    plt.savefig(ARTIFACTS / 'figures/detection_examples.png', dpi=150)
    plt.close()

plt.figure(figsize=(8,5))
plt.plot(['V1 (0.3)', 'V2 (0.7)'], [avg_v1[0], avg_v2[0]], 'o-', label='Precision')
plt.plot(['V1 (0.3)', 'V2 (0.7)'], [avg_v1[1], avg_v2[1]], 's-', label='Recall')
plt.ylabel('Score')
plt.title('Precision/Recall vs threshold')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(ARTIFACTS / 'figures/detection_metrics.png', dpi=150)
plt.close()

print(f"V1: P={avg_v1[0]:.3f}, R={avg_v1[1]:.3f}, mIoU={avg_v1[2]:.3f}")
print(f"V2: P={avg_v2[0]:.3f}, R={avg_v2[1]:.3f}, mIoU={avg_v2[2]:.3f}")

results.append({
    'experiment_id': 'V1', 'task': 'detection', 'dataset': DATASET_B, 'seed': SEED,
    'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': '-', 'lr': '-',
    'epochs_trained': 0, 'best_val_accuracy': None, 'test_accuracy': None,
    'precision': float(avg_v1[0]), 'recall': float(avg_v1[1]), 'mean_iou': float(avg_v1[2]), 
    'notes': 'threshold=0.3'
})
results.append({
    'experiment_id': 'V2', 'task': 'detection', 'dataset': DATASET_B, 'seed': SEED,
    'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': '-', 'lr': '-',
    'epochs_trained': 0, 'best_val_accuracy': None, 'test_accuracy': None,
    'precision': float(avg_v2[0]), 'recall': float(avg_v2[1]), 'mean_iou': float(avg_v2[2]), 
    'notes': 'threshold=0.7'
})



=== Часть B: Detection ===
Загрузка модели Faster R-CNN...


NameError: name 'fasterrcnn_resnet50_fpn' is not defined

In [ ]:
# 9. Сохранение результатов
df = pd.DataFrame(results)
df.to_csv(ARTIFACTS / 'runs.csv', index=False)
print("\n" + "="*50)
print("Итоговая таблица:")
print(df[['experiment_id','task','model_summary','best_val_accuracy','test_accuracy','precision','recall','mean_iou']].to_string(index=False))

print("\nПроверка артефактов:")
files = ['runs.csv', 'best_classifier.pt', 'best_classifier_config.json',
         'figures/classification_curves_best.png', 'figures/classification_compare.png',
         'figures/augmentations_preview.png', 'figures/detection_examples.png', 'figures/detection_metrics.png']
for f in files:
    status = "✓" if (ARTIFACTS / f).exists() else "✗"
    print(f"{status} {f}")

print("\n=== Готово! ===")